# Title

In [ ]:
"""
Compound Tracker

Iterative Compound-Dedupliction tool.

Tracks compounds by (name, SMILES) and prevents duplicate entries based on the Canonical SMILES 
(the chemically meaningful identifier), while still keeping record of every name used to refer to that molecle.

"""

# Set Up

In [17]:
!pip install rdkit

In [18]:
import json
import os
from typing import Optional

In [19]:
try: 
    from rdkit import Chem
except ImportError as e:
    raise ImportError(
        "RDKit is required for this script. Install with 'pip install rdkit'"
        "(or '!pip install rdkit' in a Jupyter notebook) and re-run."
    ) from e

# Function

In [21]:
class CompoundTracker:
    """
    Tracks a list of unique compounds, keyed internally by canonical SMILES.

    Each stored entry is a dict:
        {
            "canonical_smiles": str, 
            "original_smiles": str, # SMILES first used to add this compound.
            "name": str, # first name given to it.
            "synonyms": [str, ...] # other names later linked to same SMILES.
        }

    """
    def __init__(self, filepath: Optional[str] = None):
        """
        filepath: optional path to a JSON file used to persist the list across sessions.
        If the file already exists, it is loaded on init.
        """
        self.filepath = filepath
        self.compounds = {} #canonical smiles -> entry dict
        if filepath and os.path.exists(filepath):
            self.load(filepath)
    
    @staticmethod
    def canonicalise(smiles: str) -> Optional[str]:
        """
        Convert a SMILES string to RDKit's canonical form, so two differently-written SMILES for the same molecule are recognised as identical.
        Returns None if RDKit cannot parse the string.
        
        """
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    
    def add_compound(self, name: str, smiles: str) -> str:
        """
        Attempt to add a compound.
        Returns one of:
            'added' - new, unique compound added.
            'duplicate_same_name' - exact duplicate, nothing changed.
            'duplicate_new_name' - same molecule, new name -> added as synonym.
            'invalid_smiles' - RDKit could not parse the SMILES.
        """
        canonical = self.canonicalise(smiles)
        if canonical is None:
            print(f"[INVALID] Could not parse SMILES for '{name}': {smiles}")
            return "invalid_smiles"
        
        if canonical not in self.compounds:
            self.compounds[canonical] = {
                "canonical_smiles": canonical,
                "original_smiles": smiles,
                "name": name,
                "synonyms": []
            }
            print(f"[ADDED] '{name}' is new. Added to the list")
            if self.filepath:
                self.save()
            return "added"
        
        entry = self.compounds[canonical]
        existing_names = [entry["name"]] + entry["synonyms"]
        if name in existing_names:
            print(f"[DUPLICATE] '{name}' already exists with this SMILES. No change made")
            return "duplicate_same_name"
        
        entry["synonyms"].append(name)
        print(
            f"[SYNONYM] This SMILES already exists as '{entry['name']}'. " 
            f"Added '{name}' as a synonym."
        )
        if self.filepath:
            self.save()
        return "duplicate_new_name"
    
    def save(self, filepath: Optional[str] = None):
        path = filepath or self.filepath
        if not path:
            raise ValueError("No filepath provided for saving.")
        with open(path, 'w') as f:
            json.dump(self.compounds, f, indent=2)
    
    def load(self, filepath: Optional[str] = None):
        path = filepath or self.filepath
        if not path or not os.path.exists(path):
            return
        with open(path, 'r') as f:
            self.compounds = json.load(f)

    def to_list(self):
        """
        Flat list of compound dicts (handy for converting to a pandas DataFrame).
        
        """
        return list(self.compounds.values())

# Self Test

In [8]:
if __name__ == "__main__":
    # Example usage / quick self-test
    tracker = CompoundTracker(filepath="compounds.json")
    tracker.add_compound("Ethanol", "CCO")
    tracker.add_compound("Ethyl alcohol", "OCC") # same molecule, different SMILES/name - > synonym
    tracker.add_compound("Ethanol", "CCO") # exact duplicate -> no change
    tracker.add_compound("Methanol", "CO") # new, unique compound
    tracker.add_compound("not a real smiles", "????") # invalid SMILES

    print("\nCurrent list of compounds:")
    for entry in tracker.to_list():
        print(entry)

[ADDED] 'Ethanol' is new. Added to the list
[SYNONYM] This SMILES already exists as 'Ethanol'. Added 'Ethyl alcohol' as a synonym.
[DUPLICATE] 'Ethanol' already exists with this SMILES. No change made
[ADDED] 'Methanol' is new. Added to the list
[INVALID] Could not parse SMILES for 'not a real smiles': ????

Current list of compounds:
{'canonical_smiles': 'CCO', 'original_smiles': 'CCO', 'name': 'Ethanol', 'synonyms': ['Ethyl alcohol']}
{'canonical_smiles': 'CO', 'original_smiles': 'CO', 'name': 'Methanol', 'synonyms': []}


[17:20:21] SMILES Parse Error: syntax error while parsing: ????
[17:20:21] SMILES Parse Error: check for mistakes around position 1:
[17:20:21] ????
[17:20:21] ^
[17:20:21] SMILES Parse Error: Failed parsing SMILES '????' for input: '????'


## Clear Test Compounds from List

In [32]:
import os
if os.path.exists("compounds.json"):
    os.remove("compounds.json")

tracker = CompoundTracker(filepath="compounds.json")  # new object, nothing loaded into it

for entry in tracker.to_list():
    print(entry)

# Use Function

## Add Compound

In [31]:
status = tracker.add_compound("Compound X", "CCO")

if status == "added":
    print("It's genuinely new — go ahead and process it.")
elif status == "duplicate_new_name":
    print("Same molecule, just a new name — you might want to double check which name is correct.")

[SYNONYM] This SMILES already exists as 'Ethanol,'. Added 'Compound X' as a synonym.
Same molecule, just a new name — you might want to double check which name is correct.


## Check List

In [33]:
for entry in tracker.to_list():
    print(entry)